# WINGS3 — Evaluation v1 → v2

Run in JupyterLab workbench **wings3-demo**, project `my-first-model`.

**Red thread:** traces showed what happened; these cells show whether a prompt change moved the score.

**Say this first:** `contains_expected` is a **substring** check on **four** math rows. It is not an LLM-as-judge and not a production SLO. Rehearsal was 25% → 50%; `has_numeric_result` was already 100% on both.

On stage, stop at each **SHOW:** comment. Skip the v1 run cell if rehearsal already logged `v1-baseline`.


## 0. Optional: git pull

JupyterLab root is this clone. Skip on stage if already current. Cluster must reach GitHub.


In [ ]:
# Optional: update from GitHub. Skip on stage if already current.
!git pull --ff-only


## 1b. Install deps if `langchain_core` is missing

Run this **before** the env cell on a fresh kernel (env imports langchain/mlflow). Skip if those imports already work. Re-run after a workbench restart — the venv is not on the PVC. `--extra-index-url` is required: the RHOAI 3.4 RHAI index has langgraph 1.x only and may not have `langchain-core`.


In [ ]:
# Kernel venv is not on the PVC. Skip if `import langchain_core` already works.
%pip install -r ../agent-tracing/requirements.txt --extra-index-url https://pypi.org/simple


## 1. Workbench env

Tracking URI is injected when the notebook has `opendatahub.io/mlflow-instance`. You still set `MLFLOW_WORKSPACE`.


In [ ]:
import importlib
import logging
import math
import os
import sys
import warnings
from pathlib import Path

import mlflow
from langchain_core.tools import tool
from mlflow.genai.scorers import scorer

warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

os.environ.setdefault("MLFLOW_WORKSPACE", "my-first-model")
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", "wings3-agent-eval")
os.environ.setdefault("MAAS_API_KEY", "unused")
os.environ.setdefault("MAAS_MODEL", "llama-32-3b-instruct")
os.environ.setdefault(
    "MAAS_BASE_URL",
    "http://llama-32-3b-instruct-predictor.my-first-model.svc.cluster.local:8080/v1",
)

demo = Path("../agent-tracing").resolve()
if not (demo / "traced_agent.py").is_file():
    raise FileNotFoundError(f"expected traced_agent.py next to notebooks: {demo}")
sys.path.insert(0, str(demo))
traced_agent = importlib.import_module("traced_agent")
create_agent_graph = traced_agent.create_agent_graph
get_config_from_env = traced_agent.get_config_from_env

for k in (
    "MLFLOW_TRACKING_URI",
    "MLFLOW_WORKSPACE",
    "MLFLOW_K8S_INTEGRATION",
    "MLFLOW_TRACKING_AUTH",
):
    print(f"{k}={os.environ.get(k)}")


## 2. SHOW: prompts

v1 is vague. v2 requires the calculator and a numeric answer — that is what the substring scorer rewards.


In [ ]:
# SHOW: v1 is vague — the model may skip the tool or omit the number
PROMPTS = {
    "v1": "You are a helper. Answer briefly.",
    # SHOW: v2 requires the calculator and a numeric result in the text
    "v2": (
        "You are a precise math assistant. Always use the calculator tool for arithmetic. "
        "State the numeric result clearly in your answer."
    ),
}

print("=== v1 (baseline) ===")
print(PROMPTS["v1"])
print()
print("=== v2 (improved) ===")
print(PROMPTS["v2"])


## 3. SHOW: four-row dataset

`contains_expected` looks for `expected_answer` as a **substring** of the model output.


In [ ]:
# SHOW: four rows only. The scorer looks for expected_answer as a substring.
EVAL_DATASET = [
    {"inputs": {"user_message": "What is 25 * 17 + 89?"}, "expectations": {"expected_answer": "514"}},
    {"inputs": {"user_message": "Calculate 256 divided by 16"}, "expectations": {"expected_answer": "16"}},
    {"inputs": {"user_message": "What is the square root of 144?"}, "expectations": {"expected_answer": "12"}},
    {"inputs": {"user_message": "Multiply 33 by 3 and add 1"}, "expectations": {"expected_answer": "100"}},
]

row = EVAL_DATASET[1]
print("Example:", row["inputs"]["user_message"])
print("contains_expected looks for:", row["expectations"]["expected_answer"])


## 4. SHOW: substring scorer

This is the whole metric. Not an LLM-as-judge. Not a production SLO.


In [ ]:
@tool
def calculator(operation: str, a: float, b: float | None = None) -> str:
    """Arithmetic tool — same calculator as Act 2. For sqrt, pass only a."""
    ops = {
        "add": lambda x, y: x + y,
        "subtract": lambda x, y: x - y,
        "multiply": lambda x, y: x * y,
        "divide": lambda x, y: x / y if y else "Error",
        "sqrt": lambda x, _: math.sqrt(x),
        "power": lambda x, y: x**y,
    }
    if operation not in ops:
        return f"Unknown operation {operation}"
    if operation != "sqrt" and b is None:
        return f"Error: {operation} needs two numbers a and b"
    result = ops[operation](a, b)
    return f"Result: {result}" if operation != "sqrt" else f"sqrt({a}) = {result}"


# SHOW: substring scorer — True if the expected digits appear anywhere in the output
@scorer
def contains_expected(inputs: dict, outputs: str, expectations: dict) -> bool:
    if outputs is None or expectations is None:
        return False
    expected = str(expectations.get("expected_answer", ""))
    return expected.lower() in str(outputs).lower()  # SHOW: this is the whole metric


@scorer
def has_numeric_result(outputs: str) -> bool:
    if outputs is None:
        return False
    return any(c.isdigit() for c in str(outputs))


print("contains_expected example: expected '16' in 'The result is 16.0' ->", "16" in "The result is 16.0")


## 5. SHOW: `mlflow.genai.evaluate()`

Dataset + predict function + scorers. Run this cell to **define** `run_eval`; the next cells actually score v1 and v2.


In [ ]:
_agent = None
_agent_version = None
_run_version = "v1"


def get_agent(version: str):
    global _agent, _agent_version
    if _agent is None or _agent_version != version:
        prompt = PROMPTS[version]
        print(f"System prompt ({version}):\n{prompt}\n")
        _agent = create_agent_graph(
            get_config_from_env(), tools=[calculator], system_prompt=prompt
        )
        _agent_version = version
    return _agent


def predict_fn(user_message: str) -> str:
    try:
        result = get_agent(_run_version).invoke(
            {"messages": [{"role": "user", "content": user_message}]}
        )
        return result["messages"][-1].content
    except Exception as exc:
        return f"Error: {exc}"


def run_eval(version: str) -> dict:
    global _run_version, _agent, _agent_version
    _run_version = version
    _agent = None
    _agent_version = None

    k8s = os.environ.get("MLFLOW_K8S_INTEGRATION", "").lower() == "true"
    if not k8s and not os.environ.get("MLFLOW_TRACKING_TOKEN"):
        raise RuntimeError("Run in the workbench or set MLFLOW_TRACKING_TOKEN")
    uri = os.environ.get("MLFLOW_TRACKING_URI")
    if not uri:
        raise RuntimeError("MLFLOW_TRACKING_URI is not set")
    if not os.environ.get("MLFLOW_WORKSPACE"):
        raise RuntimeError("Set MLFLOW_WORKSPACE=my-first-model")

    run_name = f"{version}-{'baseline' if version == 'v1' else 'improved-prompt'}"
    mlflow.set_tracking_uri(uri)
    mlflow.set_experiment(os.environ.get("MLFLOW_EXPERIMENT_NAME", "wings3-agent-eval"))
    mlflow.langchain.autolog()
    get_agent(version)

    print(f"Running evaluation: {run_name} ({len(EVAL_DATASET)} examples)")
    with mlflow.start_run(run_name=run_name):
        # SHOW: this is the eval API — dataset + predict_fn + scorers
        result = mlflow.genai.evaluate(
            data=EVAL_DATASET,
            predict_fn=predict_fn,
            scorers=[contains_expected, has_numeric_result],
        )

    print("\nAggregated metrics:")
    for name, value in result.metrics.items():
        if isinstance(value, float):
            print(f"  {name}: {value:.2%}")
        else:
            print(f"  {name}: {value}")
    return result.metrics

print("Defined run_eval(version). Next: v1 (optional), then v2.")


## 6. Run v1 baseline

Skip this cell if rehearsal already logged `v1-baseline` and you are short on time.


In [ ]:
# Skip if v1-baseline already exists in MLflow and the clock is tight.
metrics_v1 = run_eval("v1")
metrics_v1


## 7. Run v2 improved prompt


In [ ]:
metrics_v2 = run_eval("v2")
metrics_v2


## 8. Compare, then open Evaluation

Standalone `/mlflow` → workspace `my-first-model` → experiment `wings3-agent-eval` → **Evaluation**.

Pick a **False** `contains_expected` row and read the output, then a **True** row.


In [ ]:
def _fmt(metrics, key):
    if not metrics:
        return "—"
    val = metrics.get(key)
    if isinstance(val, float):
        return f"{val:.0%}"
    return val if val is not None else "—"

v1 = globals().get("metrics_v1") or {}
v2 = globals().get("metrics_v2") or {}
print(f"{'metric':<28} {'v1':>8} {'v2':>8}")
print("-" * 46)
for key in sorted(set(v1) | set(v2)):
    print(f"{key:<28} {_fmt(v1, key):>8} {_fmt(v2, key):>8}")

print()
print("Open MLflow → workspace my-first-model → experiment wings3-agent-eval → Evaluation")
